In [10]:

# ============================================================
# CS 340 - Project Two: Grazioso Salvare Rescue Dashboard
# ============================================================

# Imports
import os
import base64
import pandas as pd
import plotly.express as px
import dash_leaflet as dl

from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

# Import the MongoDB CRUD module
from CRUD_Python_Module import AnimalShelter

# Configure Jupyter Dash for Codio
JupyterDash.infer_jupyter_proxy_config()

# ============================================================
# DATABASE CONNECTION
# ============================================================

username = "aacuser"
password = "Lakers17"

db = AnimalShelter(username, password)

# ============================================================
# RESCUE FILTERS
# ============================================================

# Rescue profiles based on the CS 340 project specifications.
# Confirm breed and age requirements against the instructor's
# Dashboard Specifications Document before submission.

RESCUE_FILTERS = {
    "Water Rescue": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland"
            ]
        },
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {
            "$gte": 26,
            "$lte": 156
        }
    },

    "Mountain or Wilderness Rescue": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {
            "$gte": 26,
            "$lte": 156
        }
    },

    "Disaster or Individual Tracking": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {
            "$gte": 20,
            "$lte": 300
        }
    }
}

# ============================================================
# DATABASE HELPER
# ============================================================

def get_animals(query=None):
    """Retrieve animal records from MongoDB."""

    if query is None:
        query = {}

    try:
        records = db.read(query)

        dataframe = pd.DataFrame.from_records(records)

        if "_id" in dataframe.columns:
            dataframe.drop(columns=["_id"], inplace=True)

        return dataframe

    except Exception as error:
        print("Database error:", error)
        return pd.DataFrame()


# Load initial unfiltered data
df = get_animals()

if df.empty:
    print("Warning: No animal records were returned.")
else:
    print("Successfully loaded", len(df), "animal records.")

# ============================================================
# LOGO
# ============================================================

def load_logo():
    """Load the Grazioso Salvare logo from the current folder."""

    possible_files = [
        "Grazioso Salvare Logo.png",
        "Grazioso_Salvare_Logo.png",
        "grazioso_salvare_logo.png",
        "logo.png"
    ]

    for filename in possible_files:
        if os.path.exists(filename):
            with open(filename, "rb") as image_file:
                encoded = base64.b64encode(
                    image_file.read()
                ).decode("utf-8")

            return "data:image/png;base64," + encoded

    return None


logo_source = load_logo()

# ============================================================
# DASHBOARD APPLICATION
# ============================================================

app = JupyterDash(__name__)

# Dashboard styling
PAGE_STYLE = {
    "fontFamily": "Arial, sans-serif",
    "backgroundColor": "#f4f7fb",
    "padding": "20px",
    "minHeight": "100vh"
}

HEADER_STYLE = {
    "textAlign": "center",
    "color": "#17365d"
}

CARD_STYLE = {
    "backgroundColor": "white",
    "padding": "15px",
    "borderRadius": "10px",
    "marginTop": "15px",
    "boxShadow": "0 2px 6px rgba(0,0,0,0.08)"
}

# ============================================================
# DASHBOARD LAYOUT
# ============================================================

app.layout = html.Div(style=PAGE_STYLE, children=[

    # Logo and title
    html.Div(
        style=HEADER_STYLE,
        children=[

            html.Img(
                src=logo_source,
                style={
                    "height": "110px",
                    "maxWidth": "100%",
                    "objectFit": "contain"
                }
            ) if logo_source else html.H2(
                "Grazioso Salvare"
            ),

            html.H1("Animal Rescue Dashboard"),

            html.H3(
                "CS 340 Project Two - Cristian Alejandro"
            ),

            html.P(
                "Austin Animal Center Rescue Candidate Database"
            )
        ]
    ),

    html.Hr(),

    # Rescue filtering controls
    html.Div(style=CARD_STYLE, children=[

        html.H3("Select Rescue Type"),

        dcc.RadioItems(
            id="filter-type",

            options=[
                {
                    "label": " Reset / Show All",
                    "value": "Reset"
                },
                {
                    "label": " Water Rescue",
                    "value": "Water Rescue"
                },
                {
                    "label": " Mountain or Wilderness Rescue",
                    "value": "Mountain or Wilderness Rescue"
                },
                {
                    "label": " Disaster or Individual Tracking",
                    "value": "Disaster or Individual Tracking"
                }
            ],

            value="Reset",

            labelStyle={
                "display": "inline-block",
                "marginRight": "20px",
                "marginBottom": "10px"
            }
        ),

        html.Div(
            id="record-count",
            style={
                "marginTop": "12px",
                "fontWeight": "bold"
            }
        )
    ]),

    # Interactive animal data table
    html.Div(style=CARD_STYLE, children=[

        html.H3("Animal Shelter Records"),

        dash_table.DataTable(
            id="datatable-id",

            columns=[
                {
                    "name": column,
                    "id": column,
                    "selectable": True
                }
                for column in df.columns
            ],

            data=df.to_dict("records"),

            page_size=10,
            page_action="native",

            sort_action="native",
            sort_mode="multi",

            filter_action="native",

            row_selectable="single",
            selected_rows=[0] if not df.empty else [],

            column_selectable="single",
            selected_columns=[],

            style_table={
                "overflowX": "auto"
            },

            style_header={
                "backgroundColor": "#17365d",
                "color": "white",
                "fontWeight": "bold"
            },

            style_cell={
                "textAlign": "left",
                "padding": "8px",
                "minWidth": "100px",
                "maxWidth": "250px",
                "overflow": "hidden",
                "textOverflow": "ellipsis"
            },

            style_data_conditional=[
                {
                    "if": {
                        "row_index": "odd"
                    },
                    "backgroundColor": "#f2f5f9"
                }
            ]
        )
    ]),

    # Charts
    html.Div(
        style={
            "display": "flex",
            "flexWrap": "wrap",
            "gap": "15px",
            "marginTop": "15px"
        },

        children=[

            html.Div(
                id="graph-id",
                style={
                    **CARD_STYLE,
                    "flex": "1 1 420px",
                    "minWidth": "0"
                }
            ),

            html.Div(
                id="map-id",
                style={
                    **CARD_STYLE,
                    "flex": "1 1 420px",
                    "minWidth": "0"
                }
            )
        ]
    )
])

# ============================================================
# CONTROLLER: FILTER DATABASE
# ============================================================

@app.callback(
    [
        Output("datatable-id", "data"),
        Output("datatable-id", "columns"),
        Output("datatable-id", "selected_rows"),
        Output("record-count", "children")
    ],
    [
        Input("filter-type", "value")
    ]
)
def update_dashboard(filter_type):

    if filter_type == "Reset":
        query = {}
    else:
        query = RESCUE_FILTERS.get(filter_type, {})

    filtered_df = get_animals(query)

    # Maintain column structure when no matches exist
    if filtered_df.empty:
        filtered_df = pd.DataFrame(columns=df.columns)

    columns = [
        {
            "name": column,
            "id": column,
            "selectable": True
        }
        for column in filtered_df.columns
    ]

    data = filtered_df.to_dict("records")

    selected_rows = [0] if data else []

    message = (
        f"Showing {len(data)} animals "
        f"| Current Filter: {filter_type}"
    )

    return data, columns, selected_rows, message


# ============================================================
# CONTROLLER: BREED DISTRIBUTION CHART
# ============================================================

@app.callback(
    Output("graph-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data")
    ]
)
def update_graphs(view_data):

    if view_data is None:
        view_data = df.to_dict("records")

    chart_df = pd.DataFrame(view_data)

    if chart_df.empty or "breed" not in chart_df.columns:
        return html.P(
            "No breed data available for this filter."
        )

    breed_counts = (
        chart_df["breed"]
        .fillna("Unknown")
        .value_counts()
        .head(10)
        .reset_index()
    )

    breed_counts.columns = ["Breed", "Count"]

    figure = px.bar(
        breed_counts,
        x="Count",
        y="Breed",
        orientation="h",
        title="Top 10 Animal Breeds",
        labels={
            "Count": "Number of Animals",
            "Breed": "Breed"
        }
    )

    figure.update_layout(
        yaxis={
            "categoryorder": "total ascending"
        },
        margin={
            "l": 20,
            "r": 20,
            "t": 60,
            "b": 20
        },
        height=450
    )

    return dcc.Graph(
        figure=figure,
        config={
            "displayModeBar": False
        }
    )


# ============================================================
# CONTROLLER: TABLE COLUMN HIGHLIGHTING
# ============================================================

@app.callback(
    Output(
        "datatable-id",
        "style_data_conditional"
    ),
    [
        Input(
            "datatable-id",
            "selected_columns"
        )
    ]
)
def update_styles(selected_columns):

    styles = [
        {
            "if": {
                "row_index": "odd"
            },
            "backgroundColor": "#f2f5f9"
        }
    ]

    for column in selected_columns or []:
        styles.append({
            "if": {
                "column_id": column
            },
            "backgroundColor": "#D2F3FF"
        })

    return styles


# ============================================================
# CONTROLLER: GEOLOCATION MAP
# ============================================================

@app.callback(
    Output("map-id", "children"),
    [
        Input(
            "datatable-id",
            "derived_virtual_data"
        ),
        Input(
            "datatable-id",
            "derived_virtual_selected_rows"
        )
    ]
)
def update_map(view_data, selected_rows):

    if view_data is None:
        view_data = df.to_dict("records")

    map_df = pd.DataFrame(view_data)

    if map_df.empty:
        return html.P(
            "No animals available to display on the map."
        )

    required_columns = [
        "location_lat",
        "location_long"
    ]

    if not all(
        column in map_df.columns
        for column in required_columns
    ):
        return html.P(
            "Location coordinates are missing."
        )

    # Choose selected row, or first available row
    row_index = 0

    if selected_rows:
        candidate = selected_rows[0]

        if 0 <= candidate < len(map_df):
            row_index = candidate

    animal = map_df.iloc[row_index]

    latitude = pd.to_numeric(
        animal.get("location_lat"),
        errors="coerce"
    )

    longitude = pd.to_numeric(
        animal.get("location_long"),
        errors="coerce"
    )

    if pd.isna(latitude) or pd.isna(longitude):
        return html.P(
            "The selected animal has no valid coordinates."
        )

    if not (-90 <= latitude <= 90 and
            -180 <= longitude <= 180):
        return html.P(
            "The selected animal has invalid coordinates."
        )

    animal_name = str(
        animal.get("name", "Unknown")
    )

    animal_breed = str(
        animal.get("breed", "Unknown")
    )

    return html.Div(children=[

        html.H3("Animal Geolocation"),

        html.P(
            f"Selected Animal: {animal_name}"
        ),

        dl.Map(
            center=[
                float(latitude),
                float(longitude)
            ],
            zoom=11,
            style={
                "width": "100%",
                "height": "400px"
            },
            children=[

                dl.TileLayer(),

                dl.Marker(
                    position=[
                        float(latitude),
                        float(longitude)
                    ],
                    children=[

                        dl.Tooltip(animal_breed),

                        dl.Popup([
                            html.H4("Animal Information"),
                            html.P(
                                f"Name: {animal_name}"
                            ),
                            html.P(
                                f"Breed: {animal_breed}"
                            )
                        ])
                    ]
                )
            ]
        )
    ])


# ============================================================
# START DASHBOARD
# ============================================================

print("Starting Grazioso Salvare Dashboard...")

app.run_server(
    mode="inline",
    port=8050,
    debug=False,
    height=1200
)


Successfully loaded 10001 animal records.
Starting Grazioso Salvare Dashboard...


 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [19/Oct/2026 00:34:52] "GET /_alive_ad0c0ec7-8180-41b0-93bf-5c3ba346fa01 HTTP/1.1" 200 -


127.0.0.1 - - [19/Oct/2026 00:34:52] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 00:34:53] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 00:34:53] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 00:34:53] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Oct/2026 00:34:53] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Oct/2026 00:34:53] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 00:34:54] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 00:34:54] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 00:34:55] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 00:34:56] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 00:34:57] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
127.0.0.1 - - [19/Oct/2026 00:34:57] "P